In [1]:
import os
import random
from glob import glob
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.cluster import KMeans

# ---------------------------\n# CONFIGURATION\n# ---------------------------
class Config:
    # --- PATHS ---
    # Path containing the 'SECOND_train_set' folder
    TRAIN_ROOT = "/kaggle/input/second-dataset-1" 
    
    # Path containing the 'SECOND_total_test/test' folder structure
    TEST_ROOT = "/kaggle/input/second-dataset-testing" 
    
    # --- MODE ---
    # Options: 'train' (Supervised) or 'detect_masks' (Generate Unsupervised Masks)
    MODE = "train" 
    
    # Output path for masks if MODE == 'detect_masks'
    OUT_MASKS_DIR = "./masks_train"
    
    # --- HYPERPARAMETERS ---
    EPOCHS = 20
    # Reduced batch size to 4 because VGG19 is memory intensive
    BATCH_SIZE = 4 
    LR = 1e-4
    
    # Weighted Loss Settings
    # We will give higher weight to change classes to improve detection of small changes
    CHANGE_CLASS_WEIGHT = 5.0 
    
    # Hardware
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # WORKER SETTINGS (Fixed to 0 to prevent multiprocessing errors)
    NUM_WORKERS = 0 

cfg = Config()

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"Running in mode: {cfg.MODE} on {cfg.DEVICE}")
print(f"Train Root: {cfg.TRAIN_ROOT}")
print(f"Test Root:  {cfg.TEST_ROOT}")

# ---------------------------\n# DATASET CONSTANTS & UTILS\n# ---------------------------
COLOR_TO_CLASS = {
    (255,255,255): 0,   # impervious surfaces
    (0,0,255):     1,   # water
    (128,128,128): 2,   # clutter/background
    (0,128,0):     3,   # trees
    (0,255,0):     4,   # low vegetation
    (128,0,0):     5,   # buildings
}
NUM_BASE_CLASSES = len(set(COLOR_TO_CLASS.values()))
NUM_CHANGE_CLASSES = 1 + NUM_BASE_CLASSES * NUM_BASE_CLASSES

def rgb_to_index_mask(pil_mask):
    """Convert RGB mask to class index mask."""
    arr = np.array(pil_mask.convert("RGB"), dtype=np.uint8)
    H,W,_ = arr.shape
    out = np.zeros((H,W), dtype=np.uint8)
    default_val = 2 if 2 in COLOR_TO_CLASS.values() else 0
    out.fill(default_val)
    for rgb, cls in COLOR_TO_CLASS.items():
        match = np.all(arr == np.array(rgb, dtype=np.uint8), axis=2)
        out[match] = cls
    return out

def make_change_map(label1_idx, label2_idx):
    """Create semantic change label map."""
    C = NUM_BASE_CLASSES
    out = np.zeros_like(label1_idx, dtype=np.int64)
    same = (label1_idx == label2_idx)
    out[same] = 0
    diff = ~same
    if diff.sum() > 0:
        code = (label1_idx * C + label2_idx) + 1
        out[diff] = code[diff]
    return out

# ---------------------------\n# DATASET CLASS\n# ---------------------------
class SECONDChangeDataset(Dataset):
    """
    Loads im1, im2, label1, label2 from a provided base_dir.
    """
    def __init__(self, base_dir, which="train", transform=None):
        if which == "train":
            base = os.path.join(base_dir, "SECOND_train_set")
        elif which == "test":
            base = os.path.join(base_dir, "SECOND_total_test", "test")
        else:
            raise ValueError("which must be 'train' or 'test'")
            
        self.im1_dir = os.path.join(base, "im1")
        self.im2_dir = os.path.join(base, "im2")
        self.l1_dir = os.path.join(base, "label1")
        self.l2_dir = os.path.join(base, "label2")
        
        # Validation
        for d in [self.im1_dir, self.im2_dir, self.l1_dir, self.l2_dir]:
            if not os.path.isdir(d):
                pass # Warning: In Kaggle inference some folders might be missing, strict check removed for portability
                
        self.ids = sorted([os.path.splitext(os.path.basename(p))[0] for p in glob(os.path.join(self.im1_dir, "*.png"))])
        self.transform = transform or (T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]))
        self.which = which
        self.base = base

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid = self.ids[idx]
        p1 = os.path.join(self.im1_dir, pid + ".png")
        p2 = os.path.join(self.im2_dir, pid + ".png")
        l1 = os.path.join(self.l1_dir, pid + ".png")
        l2 = os.path.join(self.l2_dir, pid + ".png")
        
        im1 = Image.open(p1).convert("RGB")
        im2 = Image.open(p2).convert("RGB")
        lab1 = Image.open(l1)
        lab2 = Image.open(l2)
        
        im1_t = self.transform(im1)
        im2_t = self.transform(im2)
        lab1_idx = rgb_to_index_mask(lab1)
        lab2_idx = rgb_to_index_mask(lab2)
        change_target = make_change_map(lab1_idx, lab2_idx)
        
        return im1_t, im2_t, torch.from_numpy(change_target).long(), pid

# ---------------------------\n# HELPER: KMEANS MASK GENERATOR (OPTIONAL)\n# ---------------------------
class ResNetFeatExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        # Keeping ResNet18 for the auxiliary tool as it is lightweight
        rn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool, 
                                      rn.layer1, rn.layer2, rn.layer3)
    def forward(self, x):
        return self.features(x)

def compute_kmeans_mask(im1_path, im2_path, device='cpu', sample_pixels=50000):
    """Compute KMeans mask for a single pair. Runs on CPU by default to save GPU memory."""
    tf = T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    im1 = tf(Image.open(im1_path).convert("RGB")).unsqueeze(0)
    im2 = tf(Image.open(im2_path).convert("RGB")).unsqueeze(0)
    
    # If using CUDA, move model and data
    proc_device = torch.device(device)
    model = ResNetFeatExtractor(pretrained=True).to(proc_device).eval()
    
    with torch.no_grad():
        f1 = model(im1.to(proc_device))
        f2 = model(im2.to(proc_device))
        
    diff = torch.abs(f2 - f1)[0]
    diff_up = F.interpolate(diff.unsqueeze(0), size=(512,512), mode='bilinear', align_corners=False)[0].cpu().numpy()
    
    C,H,W = diff_up.shape
    flat = diff_up.reshape(C, -1).T
    
    # Subsample pixels for KMeans fitting
    n = flat.shape[0]
    sample = flat[np.random.choice(n, sample_pixels, replace=False)] if n > sample_pixels else flat

    kmeans = KMeans(n_clusters=2, random_state=0, n_init=4).fit(sample)
    labels = kmeans.predict(flat).reshape(H,W)
    
    # Determine which cluster is 'change' (higher mean magnitude)
    cl_means = []
    for cl in [0,1]:
        cl_vals = flat[labels.flatten()==cl]
        if cl_vals.shape[0] == 0: cl_means.append(0.0)
        else: cl_means.append(np.linalg.norm(cl_vals, axis=1).mean())
            
    changed_cl = int(np.argmax(cl_means))
    return (labels == changed_cl).astype(np.uint8)

# ---------------------------\n# MODEL: VGG19 Segmentation (Replaces ResNet-18)\n# ---------------------------
class VGG19Segmentation(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        # 1. Load VGG19 with Batch Norm (crucial for convergence)
        weights = models.VGG19_BN_Weights.DEFAULT if pretrained else None
        vgg = models.vgg19_bn(weights=weights)
        features = list(vgg.features.children())
        
        # 2. Modify the first layer to accept 6 channels (Im1 + Im2)
        # Original is Conv2d(3, 64, k=3, p=1)
        original_conv1 = features[0]
        features[0] = nn.Conv2d(6, 64, kernel_size=3, padding=1)
        
        # Initialize new weights by averaging/copying pre-trained weights
        if pretrained:
            with torch.no_grad():
                features[0].weight[:, :3] = original_conv1.weight
                features[0].weight[:, 3:] = original_conv1.weight
                features[0].bias = original_conv1.bias

        # 3. Encoder Construction
        # We need to slice the VGG features based on MaxPool layers ('M')
        # VGG19_BN structure roughly:
        # Block1 (64) -> Pool1
        # Block2 (128) -> Pool2
        # Block3 (256) -> Pool3
        # Block4 (512) -> Pool4
        # Block5 (512) -> Pool5
        
        pool_indices = [i for i, layer in enumerate(features) if isinstance(layer, nn.MaxPool2d)]
        
        # VGG19_BN pool indices are typically: 6, 13, 26, 39, 52
        self.enc1 = nn.Sequential(*features[0:pool_indices[0]]) # 64 channels
        self.pool1 = features[pool_indices[0]]
        
        self.enc2 = nn.Sequential(*features[pool_indices[0]+1:pool_indices[1]]) # 128 channels
        self.pool2 = features[pool_indices[1]]
        
        self.enc3 = nn.Sequential(*features[pool_indices[1]+1:pool_indices[2]]) # 256 channels
        self.pool3 = features[pool_indices[2]]
        
        self.enc4 = nn.Sequential(*features[pool_indices[2]+1:pool_indices[3]]) # 512 channels
        self.pool4 = features[pool_indices[3]]
        
        self.enc5 = nn.Sequential(*features[pool_indices[3]+1:pool_indices[4]]) # 512 channels
        self.pool5 = features[pool_indices[4]]

        # 4. Decoder
        # Bottleneck processes pool5 output (512ch)
        self.center = self._decoder_block(512, 512)
        
        # Dec5: Input = Center(512) + Enc5(512) = 1024 -> Output 256
        self.dec5 = self._decoder_block(1024, 256)
        
        # Dec4: Input = Dec5(256) + Enc4(512) = 768 -> Output 128
        self.dec4 = self._decoder_block(768, 128)
        
        # Dec3: Input = Dec4(128) + Enc3(256) = 384 -> Output 64
        self.dec3 = self._decoder_block(384, 64)
        
        # Dec2: Input = Dec3(64) + Enc2(128) = 192 -> Output 64
        self.dec2 = self._decoder_block(192, 64)
        
        # Dec1: Input = Dec2(64) + Enc1(64) = 128 -> Output 64
        self.dec1 = self._decoder_block(128, 64)
        
        self.final = nn.Conv2d(64, num_classes, kernel_size=1)

    def _decoder_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        e5 = self.enc5(self.pool4(e4))
        f = self.pool5(e5)
        
        # Center
        c = self.center(f)
        
        # Decoder with Skip Connections
        d5 = F.interpolate(c, size=e5.shape[2:], mode='bilinear', align_corners=False)
        d5 = self.dec5(torch.cat([d5, e5], dim=1))
        
        d4 = F.interpolate(d5, size=e4.shape[2:], mode='bilinear', align_corners=False)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        
        d3 = F.interpolate(d4, size=e3.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = F.interpolate(d3, size=e2.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = F.interpolate(d2, size=e1.shape[2:], mode='bilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        # Final output
        out = self.final(d1)
        out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        
        return out

# ---------------------------\n# TRAINING & VALIDATION LOOPS\n# ---------------------------
def compute_mIoU(pred, target, num_classes):
    pred = pred.flatten()
    target = target.flatten()
    ious = []
    for cls in range(num_classes):
        union = np.logical_or(pred == cls, target == cls).sum()
        if union == 0:
            ious.append(np.nan)
        else:
            intersection = np.logical_and(pred == cls, target == cls).sum()
            ious.append(intersection / union)
    ious_valid = [v for v in ious if not np.isnan(v)]
    return np.mean(ious_valid) if len(ious_valid) else 0.0

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_acc, total_samples = 0.0, 0.0, 0
    
    # --- WEIGHTED LOSS SETUP ---
    # Create class weights: 1.0 for No Change, 5.0 for all Change classes
    weight_tensor = torch.ones(NUM_CHANGE_CLASSES).to(device)
    weight_tensor[1:] = cfg.CHANGE_CLASS_WEIGHT
    
    for im1, im2, target, pid in tqdm(loader, desc="Train"):
        im1, im2, target = im1.to(device), im2.to(device), target.to(device)
        x = torch.cat([im1, im2], dim=1)
        logits = model(x)
        
        loss = F.cross_entropy(logits, target, weight=weight_tensor)
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * im1.size(0)
        preds = logits.argmax(1)
        total_acc += (preds == target).float().mean().item() * im1.size(0)
        total_samples += im1.size(0)
        
    return total_loss / total_samples, total_acc / total_samples

def validate(model, loader, device):
    model.eval()
    accs, mious = [], []
    with torch.no_grad():
        for im1, im2, target, pid in tqdm(loader, desc="Val"):
            im1, im2, target = im1.to(device), im2.to(device), target.to(device)
            x = torch.cat([im1, im2], dim=1)
            logits = model(x)
            preds = logits.argmax(1).cpu().numpy()
            tgt = target.cpu().numpy()
            
            for p, t in zip(preds, tgt):
                accs.append((p == t).mean())
                mious.append(compute_mIoU(p, t, NUM_CHANGE_CLASSES))
    return float(np.mean(accs)), float(np.mean(mious))

def evaluate_binary_miou(model, loader, device):
    model.eval()
    intersection = torch.zeros(2).to(device)
    union = torch.zeros(2).to(device)
    
    with torch.no_grad():
        for im1, im2, target, _ in tqdm(loader, desc="Calculating Binary mIoU"):
            im1, im2, target = im1.to(device), im2.to(device), target.to(device)
            x = torch.cat([im1, im2], dim=1)
            logits = model(x)
            preds = logits.argmax(1)
            
            bin_preds = (preds > 0).long()
            bin_target = (target > 0).long()
            
            for cls in [0, 1]:
                inter = ((bin_preds == cls) & (bin_target == cls)).sum()
                area_pred = (bin_preds == cls).sum()
                area_target = (bin_target == cls).sum()
                u = area_pred + area_target - inter
                
                intersection[cls] += inter
                union[cls] += u

    iou_no_change = intersection[0] / (union[0] + 1e-6)
    iou_change = intersection[1] / (union[1] + 1e-6)
    binary_miou = (iou_no_change + iou_change) / 2.0
    
    return binary_miou.item() * 100, iou_no_change.item() * 100, iou_change.item() * 100

# ---------------------------\n# MAIN EXECUTION\n# ---------------------------
if __name__ == "__main__":
    if cfg.MODE == "detect_masks":
        print(">>> Generating Unsupervised Masks...")
        im1_dir = os.path.join(cfg.TRAIN_ROOT, "SECOND_train_set", "im1")
        im2_dir = os.path.join(cfg.TRAIN_ROOT, "SECOND_train_set", "im2")
        os.makedirs(cfg.OUT_MASKS_DIR, exist_ok=True)
        
        ids = sorted([os.path.splitext(os.path.basename(p))[0] for p in glob(os.path.join(im1_dir, "*.png"))])
        print(f"Found {len(ids)} pairs.")
        
        for pid in tqdm(ids, desc="Processing"):
            p1 = os.path.join(im1_dir, pid + ".png")
            p2 = os.path.join(im2_dir, pid + ".png")
            mask = compute_kmeans_mask(p1, p2, device=cfg.DEVICE, sample_pixels=30000)
            Image.fromarray((mask*255).astype('uint8')).save(os.path.join(cfg.OUT_MASKS_DIR, pid + "_mask.png"))
        print(f"Masks written to {cfg.OUT_MASKS_DIR}")

    elif cfg.MODE == "train":
        print(">>> Starting Training Pipeline with VGG19_BN (Weighted)...")
        
        # Init Datasets
        train_ds = SECONDChangeDataset(cfg.TRAIN_ROOT, which="train")
        test_ds  = SECONDChangeDataset(cfg.TEST_ROOT, which="test")
        
        # NOTE: num_workers=0 fixes the multiprocessing crash
        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, 
                                  num_workers=cfg.NUM_WORKERS, pin_memory=True)
        test_loader  = DataLoader(test_ds, batch_size=max(1, cfg.BATCH_SIZE//2), shuffle=False, 
                                  num_workers=cfg.NUM_WORKERS, pin_memory=True)
        
        print(f"Train samples: {len(train_ds)} | Test samples: {len(test_ds)}")
        
        # Init Model - UPDATED TO VGG19
        model = VGG19Segmentation(num_classes=NUM_CHANGE_CLASSES, pretrained=True).to(cfg.DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-5)
        
        best_miou = 0.0
        
        # Train Loop
        for epoch in range(cfg.EPOCHS):
            print(f"\nEpoch {epoch+1}/{cfg.EPOCHS}")
            
            train_loss, train_acc = train_one_epoch(
                model, train_loader, optimizer, cfg.DEVICE
            )
            print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
            
            val_acc, val_miou = validate(model, test_loader, cfg.DEVICE)
            print(f"Test  Acc:  {val_acc:.4f} | mIoU: {val_miou:.4f}")
            
            # Save Best
            if val_miou > best_miou:
                best_miou = val_miou
                torch.save(model.state_dict(), "best_change_vgg19.pt")
                print("--> Best Model Saved.")

        # --- RUN EVALUATION ---
        print(">>> Computing Binary Metrics for Comparison with Paper...")
        model.load_state_dict(torch.load("best_change_vgg19.pt"))
        bin_miou, iou_nc, iou_c = evaluate_binary_miou(model, test_loader, cfg.DEVICE)

        print("-" * 30)
        print(f"Binary mIoU (Paper Metric): {bin_miou:.2f}%")
        print("-" * 30)
        print(f"  - IoU (No Change):        {iou_nc:.2f}%")
        print(f"  - IoU (Change):           {iou_c:.2f}%")
        print("-" * 30)
        print("Reference Benchmark (FC-EF): 59.3%")

Running in mode: train on cuda
Train Root: /kaggle/input/second-dataset-1
Test Root:  /kaggle/input/second-dataset-testing
>>> Starting Training Pipeline with VGG19_BN (Weighted)...
Train samples: 2968 | Test samples: 1694


Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth
100%|██████████| 548M/548M [00:02<00:00, 244MB/s]



Epoch 1/20


Train:   0%|          | 0/742 [00:00<?, ?it/s]

Train Loss: 2.3487 | Acc: 0.5566


...

Train Loss: 0.5106 | Acc: 0.8781


Val:   0%|          | 0/847 [00:00<?, ?it/s]

Test  Acc:  0.8005 | mIoU: 0.1710
>>> Computing Binary Metrics for Comparison with Paper...


Calculating Binary mIoU:   0%|          | 0/847 [00:00<?, ?it/s]

------------------------------
Binary mIoU (Paper Metric): 63.99%
------------------------------
  - IoU (No Change):        81.32%
  - IoU (Change):           46.65%
------------------------------
Reference Benchmark (FC-EF): 59.3%
